# PowerPlant

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("powerplant_data.csv")

X = df.drop("PE", axis=1)
y = df["PE"]

In [2]:
X.head()

,AT,V,AP,RH
0,8.34,40.77,1010.84,90.01
1,23.64,58.49,1011.40,74.20
2,29.74,56.90,1007.15,41.91
3,19.07,49.69,1007.22,76.79
4,11.80,40.66,1017.13,97.20


In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [4]:
from sklearn.preprocessing import StandardScaler

sc = StandardScaler()

X_train_sc = sc.fit_transform(X_train)
X_test_sc = sc.transform(X_test)

In [5]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_train_tensor = torch.tensor(X_train_sc, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)

X_test_tensor = torch.tensor(X_test_sc, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

In [6]:
from torch.utils.data import DataLoader, TensorDataset

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

# ANN 

In [7]:
class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(X_train.shape[1], 6),
            nn.ReLU(),

            nn.Linear(6, 6),
            nn.ReLU(),

            nn.Linear(6, 1),
        )
    def forward(self, x):
        return self.model(x)

In [8]:
import torch.optim as optim
model = ANN().to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters())

In [9]:
train_loss = []
val_loss = []

best_val_loss = float("inf")
epochs = 100

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for xb, yb in train_loader:     # xb -> feat of 1 batch, yb -> label of 1 batch
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        output = model(xb)

        loss = criterion(output, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()     #loss ia s tensor => py float

    epoch_train_loss = running_loss / len(train_loader)
    train_loss.append(epoch_train_loss)

    #validation
    model.eval()
    running_val_loss = 0.0

    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            output = model(xb)

            loss = criterion(output, yb)
            running_val_loss += loss

    epoch_val_loss = running_val_loss/ len(test_loader)
    val_loss.append(epoch_val_loss)

    print(f"epoch = {epoch+1}/{epochs} => train loss = {epoch_train_loss} & val loss = {epoch_val_loss}")

    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), "best_model.pt")

epoch = 1/100 => train loss = 205393.26966145833 & val loss = 202736.203125
epoch = 2/100 => train loss = 193445.68326822916 & val loss = 178430.34375
epoch = 3/100 => train loss = 152991.03160807292 & val loss = 123355.265625
epoch = 4/100 => train loss = 91780.16416015624 & val loss = 63227.43359375
epoch = 5/100 => train loss = 44056.78143717448 & val loss = 31176.431640625
epoch = 6/100 => train loss = 25614.54069824219 & val loss = 22331.234375
epoch = 7/100 => train loss = 20256.961311848958 & val loss = 18472.37109375
epoch = 8/100 => train loss = 16760.192826334634 & val loss = 15106.3564453125
epoch = 9/100 => train loss = 13532.118853759766 & val loss = 11854.7744140625
epoch = 10/100 => train loss = 10403.366835530598 & val loss = 8886.6787109375
epoch = 11/100 => train loss = 7613.685397338867 & val loss = 6348.07568359375
epoch = 12/100 => train loss = 5342.497172037761 & val loss = 4319.830078125
epoch = 13/100 => train loss = 3517.6663599650064 & val loss = 2739.53637695